# 01 — Collecting public article metadata with Selenium

**Purpose.** Demonstrate a cautious, resumable workflow for collecting public article metadata and text into SQLite.

> **Before running:** check the website's current terms, robots directives, copyright rules and applicable law. Use only if permitted. This notebook does not run automatically on import. Website markup changes frequently; inspect and update the selectors before use. Do not commit scraped article text or the resulting database unless you have permission to redistribute it.

## 1. Setup

The scraper uses Selenium because the source page may load articles dynamically. Keep request frequency low, stop if the site signals errors, and avoid bypassing access controls.

In [ ]:
from __future__ import annotations

import json
import random
import sqlite3
import time
from pathlib import Path
from typing import Any

from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

DB_PATH = Path("../data/ScrapedNiusData.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
START_URL = "https://www.nius.de/all-news"
MAX_ARTICLES = 400
MIN_DELAY_SECONDS = 20
MAX_DELAY_SECONDS = 30


## 2. Create a local SQLite table

The schema stores metadata separately from the article body. The database is local and intentionally excluded from version control by `.gitignore`.

In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS articles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    category TEXT,
    title TEXT NOT NULL UNIQUE,
    published_at TEXT,
    author TEXT,
    article_text TEXT,
    highlights TEXT,
    metadata_json TEXT,
    source_url TEXT,
    collected_at TEXT DEFAULT CURRENT_TIMESTAMP
)
"""

with sqlite3.connect(DB_PATH) as conn:
    conn.execute(SCHEMA)

print(f"Database ready: {DB_PATH.resolve()}")


## 3. Configure the browser

Use headless mode for reproducibility. The CSS selectors below are examples based on the original coursework and may need maintenance. Test on one page before collecting a larger sample.

In [ ]:
def build_driver() -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1440,1000")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=options)

# Explicit opt-in: uncomment only after reviewing the website rules and selectors.
# driver = build_driver()
# driver.get(START_URL)
# wait = WebDriverWait(driver, 15)


## 4. Extraction template

The selectors are brittle by nature: validate them against the current HTML and record the date of collection. This template deliberately does not start a full scrape automatically.

In [ ]:
def extract_article(driver: webdriver.Chrome, url: str) -> dict[str, Any]:
    """Extract one article using selectors that must be checked against current markup."""
    driver.get(url)
    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

    def text_or_empty(by: By, selector: str) -> str:
        try:
            return driver.find_element(by, selector).text.strip()
        except NoSuchElementException:
            return ""

    title = text_or_empty(By.TAG_NAME, "h1")
    category = text_or_empty(By.CSS_SELECTOR, "[class*='ArticleHeader'][class*='category']")
    published_at = text_or_empty(By.CSS_SELECTOR, "[class*='ArticleAside'][class*='date']")
    author = text_or_empty(By.CSS_SELECTOR, "[class*='authorName']")
    paragraphs = driver.find_elements(By.CSS_SELECTOR, "[class*='ArticleMain'][class*='body'] p")
    article_text = "\n".join(p.text.strip() for p in paragraphs if p.text.strip())
    highlights = " ".join(
        e.text.strip() for e in driver.find_elements(By.CSS_SELECTOR, "h2, p strong") if e.text.strip()
    )
    metadata = {}
    for element in driver.find_elements(By.TAG_NAME, "meta"):
        key = element.get_attribute("name") or element.get_attribute("property")
        value = element.get_attribute("content")
        if key and value:
            metadata[key] = value
    return {
        "category": category, "title": title, "published_at": published_at,
        "author": author, "article_text": article_text, "highlights": highlights,
        "metadata_json": json.dumps(metadata, ensure_ascii=False), "source_url": url,
    }


def save_article(article: dict[str, Any], db_path: Path = DB_PATH) -> bool:
    """Save one article; duplicate titles are ignored. Returns True when inserted."""
    sql = """INSERT OR IGNORE INTO articles
    (category, title, published_at, author, article_text, highlights, metadata_json, source_url)
    VALUES (:category, :title, :published_at, :author, :article_text, :highlights, :metadata_json, :source_url)"""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.execute(sql, article)
        return cursor.rowcount == 1


## 5. Responsible collection checklist

- Confirm permission and current site rules before running.
- Collect only the fields needed for the research question.
- Keep a conservative delay between article requests; stop on errors or access restrictions.
- Store the database outside the public repository.
- Document selectors, collection date, sample limitations and any missing values.
- Never describe the sample as representative unless the sampling design supports that claim.